# 1. Identificação e Descrição do Problema

Título: Reconhecimento de 12 selos de mão do anime Naruto.

Integrantes: Luiz Cristóvão Rezende Poderoso, Jean Fonseca Santos e Ivanilton Vieira dos Santos.

Fonte dos Dados: [Naruto Hand Sign Dataset](https://www.kaggle.com/datasets/vikranthkanumuru/naruto-hand-sign-dataset). 

Objetivo: Reconhecer os 12 selos de mão do anime Naruto em imagens.

Atributo-Alvo: Categoria do selo de mão.

Atributos Preditivos: Matriz de pixels da imagem.

Tipo da Tarefa: Classificação.


# 2. Compreensão dos Dados



Primeiro, é necessário baixar o dataset. Para realizar essa tarefa, foi utilizada a biblioteca `gdown` que permite baixar arquivos públicos no Google Drive sem autenticação. O próprio dataset divide os dados para treinamento e teste, os quais foram unificados em uma única pasta no Drive. Outra diferença em relação ao dataset do drive do original é a exclusão completa de um dos selos de mão 'zero', o qual não pertence ao anime Naruto, sendo uma criação do autor dos dados.

In [ ]:
%pip install gdown

In [ ]:
DATASET_URL = "https://drive.google.com/uc?id=1wO8Et-vtRGRZVXs5DODhXbyGHvgcUU3p"
DATASET_OUTPUT = "dataset.zip"

In [ ]:
!gdown "{DATASET_URL}" -O "{DATASET_OUTPUT}"

Para extrair a pasta:

In [ ]:
import shutil
shutil.unpack_archive(DATASET_OUTPUT)

Para apagar arquivos/pastas remanescentes:

In [ ]:
import os
import shutil

def delete_useless():
    if DATASET_OUTPUT in os.listdir(os.path.curdir):
        os.remove(DATASET_OUTPUT)

    if "__MACOSX" in os.listdir(os.path.curdir):
        shutil.rmtree("__MACOSX")

delete_useless()

Instale as bibliotecas necessárias para análise e comparação das imagens:

In [ ]:
%pip install Pillow ImageHash

Agora, importamos as bibliotecas e declararemos as constantes necessárias:

In [ ]:
import os
from PIL import Image
import imagehash

DATASET_PATH = os.path.join("dataset")
SIGNALS = ["bird", "boar", "dog", "dragon", "hare", "horse", "monkey", "ox", "ram", "rat", "snake", "tiger"]
IMG_DIFFERENCE_LIMIT = 5

O objetivo do trecho seguinte é obter as informações acerca dos formatos das imagens, resolução e, utilizando a biblioteca `imagehash`, verificar se a imagem é duplicada ou extremamente semelhante a outra analisada anteriormente (Distância de Hamming <= 5). Também são armazenadas a quantidade de imagens úteis de cada selo de mão afim de analisar, posteriormente, o balanceamento dos dados. O output "ok" é mostrado quando os dados terminarem de ser colhidos. Esse trecho pode levar um tempo a mais para ser executado.

In [ ]:
signals = dict()

for signal in os.listdir(DATASET_PATH):
    if (signal not in SIGNALS): 
        continue

    signal_path = os.path.join(DATASET_PATH, signal)

    signals[signal] = dict()
    signals[signal]["resolutions_count"] = dict()
    signals[signal]["png_count"] = 0
    signals[signal]["jpg_count"] = 0
    signals[signal]["total_count"] = 0
    signals[signal]["duplicates_count"] = 0
    signals[signal]["simmilar_count"] = 0

    seen_hashes = set()

    for img_filename in os.listdir(signal_path):
        lower_img_name = img_filename.lower()

        # formato
        if lower_img_name.endswith(".png"): signals[signal]["png_count"] += 1
        elif lower_img_name.endswith(".jpg") or lower_img_name.endswith(".jpeg"): signals[signal]["jpg_count"] += 1
        signals[signal]["total_count"] += 1

        img_path = os.path.join(signal_path, img_filename)
        try:
            with Image.open(img_path) as img:
                # resoluções
                res = f"{img.size[0]}x{img.size[1]}"
                
                if res not in signals[signal]["resolutions_count"]: signals[signal]["resolutions_count"][res] = 0
                signals[signal]["resolutions_count"][res] += 1
                
                # duplicatas / semelhantes
                img_hash = imagehash.phash(img)
                
                is_duplicate = False
                is_similar = False

                for seen_hash in seen_hashes:
                    difference = img_hash - seen_hash
                    
                    if difference == 0:
                        is_duplicate = True
                        break
                    elif difference <= IMG_DIFFERENCE_LIMIT:
                        is_similar = True
                
                if is_duplicate:
                    signals[signal]["duplicates_count"] += 1
                elif is_similar:
                    signals[signal]["simmilar_count"] += 1
                else:
                    seen_hashes.add(img_hash)
        except (FileNotFoundError, OSError):
            pass

    signals[signal]["usable_count"] = signals[signal]["total_count"] - signals[signal]["duplicates_count"] - signals[signal]["simmilar_count"]

print("ok")

[OPCIONAL] Caso deseje visualizar relatórios para imagens de cada selo de mão, execute a célula a seguir.

In [ ]:
for signal in signals:
    print(f"{'-' * 25}\nSelo: {signal}\nTotal de Imagens: {signals[signal]["total_count"]}")

    print("\n---RESOLUÇÕES---")
    for res in signals[signal]["resolutions_count"]:
        length = signals[signal]["resolutions_count"][res]
        print(f"{res}: {length}")

    other_formats_count = signals[signal]["total_count"] - signals[signal]["png_count"] - signals[signal]["jpg_count"]

    print(f"\n---FORMATO---\nImagens PNG: {signals[signal]["png_count"]}\nImagens JPG: {signals[signal]["jpg_count"]}\nOutros formatos: {other_formats_count}")
    print(f"\n---GERAL---\nDuplicatas: {signals[signal]["duplicates_count"]}\nSemelhantes: {signals[signal]["simmilar_count"]}\nVálidas: {signals[signal]["usable_count"]}\n{'-' * 25}\n")

Para obter as informações somadas/gerais:

In [ ]:
total_count = sum(info['total_count'] for info in signals.values())
png_count = sum(info['png_count'] for info in signals.values())
jpg_count = sum(info['jpg_count'] for info in signals.values())
duplicates_count = sum(info['duplicates_count'] for info in signals.values())
simmilar_count = sum(info['simmilar_count'] for info in signals.values())
usable_count = sum(info['usable_count'] for info in signals.values())

usable_dict = dict()

for signal in signals:
    usable_dict[signal] = signals[signal]["usable_count"]

resolutions_dict = dict()

for info in signals.values():
    for resolution in info['resolutions_count']:
        resolutions_dict[resolution] = resolutions_dict.get(resolution, 0) + info['resolutions_count'][resolution]

[OPCIONAL] Por último, para obter o relatório geral acerca das imagens:

In [ ]:
print(f"{'-' * 25}\nTotal\nTotal de Imagens: {total_count}")

print("\n---RESOLUÇÕES---")
for res in resolutions_dict:
    length = resolutions_dict[res]
    print(f"{res}: {length}")
    
print(f"\n---FORMATO---\nImagens PNG: {png_count}\nImagens JPG: {jpg_count}\nOutros formatos: {total_count - png_count - jpg_count}")

max_signal = max(usable_dict, key=usable_dict.get)
min_signal = min(usable_dict, key=usable_dict.get)

print(f"\n---DISTRIBUIÇÃO DO ATRIBUTO-ALVO (IMAGENS ÚTEIS)---")
print(f"Desbalanceamento: o sinal {max_signal} possui a maior quantidade de imagens úteis: {usable_dict[max_signal]}. Já {min_signal} possui a menor quantidade, com {usable_dict[min_signal]}.\nA quantidade de imagens válidas por sinal é:")
for signal, count in usable_dict.items():
    print(f"{signal}: {count} imagens")

print(f"\n---GERAL---\nDuplicatas: {duplicates_count}\nSemelhantes: {simmilar_count}\nUtilizáveis: {usable_count}\n{'-' * 25}\n")

O relatório final aponta a existência de 2041 imagens no dataset original. As imagens estão em 3 resoluções diferentes: `640x480`, `1280x720` e `3264x1840`, majoritariamente (1620 imagens) em `1280x720`. Ademais, as imagens, em sua maioria (1619 imagens), são `PNG`, já o restante em `JPG`. Além disso, há uma discrepência considerável entre as imagens úteis dos sinais: enquanto `dog` e `boar` possuem 48, `rat` tem 24. Por fim, o dataset possui uma quantidade preocupante de imagens duplicadas (Distância de Hamming igual a 0): 420, e uma ainda mais preocupante de semelhantes (Distância de Hamming menor ou igual a 5): 1192, restando apenas 429 imagens utilizáveis Os problemas apresentados aqui serão tratados no pré-processamento dos dados.

# 3. Analise Exploratória

Abaixo realizamos a instalação e importação do matplotlib para gerar os gráficos.

In [ ]:
%pip install matplotlib

In [ ]:
import matplotlib.pyplot as plt

#### Distruibuição de imagens úteis por selo de mão

In [ ]:
plt.figure(figsize=(10, 5))
plt.bar(usable_dict.keys(), usable_dict.values())
plt.title('Distribuição de Imagens Úteis por Selo de Mão')
plt.xlabel('Selos de Mão (Classes)')
plt.ylabel('Quantidade de Imagens')
plt.xticks(rotation=45)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

A visualização do histograma acima evidencia a disparidade entre a frequência de diferentes selos de mão. O desbalanceamento é tal que algumas classes possuem o dobro de imagens úteis do que outras. Devido a esse fator e à baixa quantidade de imagens, a adição de novas imagens será realizada no dataset.

#### Relação entre atributos e alvo

In [ ]:
def plot_by_signal_images(dataset_path = DATASET_PATH, cmap="viridis"):
    fig, axes = plt.subplots(3, 4, figsize=(15, 10))
    axes = axes.flatten()

    for i, signal in enumerate(SIGNALS):
        signal_path = os.path.join(dataset_path, signal)
        
        for img_name in os.listdir(signal_path):
            if img_name.lower().endswith(('.png', '.jpg', '.jpeg')):
                img_path = os.path.join(signal_path, img_name)
                img = Image.open(img_path)
                
                axes[i].imshow(img, cmap=cmap)
                axes[i].set_title(signal)
                axes[i].axis('off')
                break

    plt.tight_layout()
    plt.show()

plot_by_signal_images()

Devido à natureza do dataset, os atributos preditivos consistem nas matrizes de pixels que compõem as imagens. Acima é possível observar uma imagem retirada do dataset de cada selo de mão. Essa plotagem demonstra a forma com que as imagens são colhidas, sendo o selo de mão ocupante de parte pequena da imagem.

#### Histogramas de intensidade de pixels

In [ ]:
import numpy as np

IMG_PER_SIGNAL = 50 

def pixels_intensity_histogram(size: int = 0):
    bins = 50
    histogram = np.zeros(bins)
    bar_limits = np.linspace(0, 256, bins + 1)

    for signal in SIGNALS:
        signal_path = os.path.join(DATASET_PATH, signal)
        
        if not os.path.exists(signal_path):
            continue
            
        count = 0
        for img_name in os.listdir(signal_path):
            if img_name.lower().endswith(('.png', '.jpg', '.jpeg')):
                img_path = os.path.join(signal_path, img_name)
                try:
                    with Image.open(img_path) as img:
                        if size > 0: img.thumbnail((size, size)) 
                        gray_img = img.convert('L')
                        
                        pixels = np.array(gray_img).flatten()
                        
                        local_count, _ = np.histogram(pixels, bins=bar_limits)
                        histogram += local_count
                        
                except (FileNotFoundError, OSError):
                    pass
                
                count += 1
                if count >= IMG_PER_SIGNAL:
                    break 

    plt.figure(figsize=(10, 5))

    bar_width = 256 / bins
    bar_center = bar_limits[:-1] + bar_width / 2

    plt.bar(bar_center, histogram, width=bar_width, color='dimgray', alpha=0.8)

    if size == 0:
        plt.title(f'Histograma de Intensidade de Pixels (Tamanho Original)')
    else:
        plt.title(f'Histograma de Intensidade de Pixels ({size}x{size})')

    plt.xlabel('Intensidade do Pixel (0 = Preto absoluto, 255 = Branco absoluto)')
    plt.ylabel('Quantidade de Pixels')
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.show()

pixels_intensity_histogram()

In [ ]:
pixels_intensity_histogram(512)

Para os histogramas de intensidade de pixels, foi escolhida uma amostragem de 50 imagens por selo. Através deles, é possível notar que o dataset apresenta picos de superexposição (concentração de pixels próximos ao limite de 255), o que pode afetar a classificação dos selos de mão por parte dos modelos. Apesar disso, ao reduzir a resolução das imagens (segundo histograma), a concentração de pixels brancos diminuiu consideravelmente, o que pode contribuir para a mitigação desse problema.

# 4. Pré-processamento

#### 4.1 Novas imagens

As imagens do dataset, além de escassas, apresentaram um baixo padrão de qualidade, sendo muitos delas frames consecutivos de um mesmo vídeo (quase nenhuma variação entre elas). Por conta disso, foi necessário adicionar novas imagens ao conjunto. Primeiro, a equipe desenvolveu um script que extrai imagens de um vídeo a cada 30 frames (veja o diretório `scripts/extracao`), já inserindo-as nas pastas corretas do dataset. Depois, os integrantes da equipe gravaram vídeos fazendo os selos de mão em diferentes ambientes com diferentes iluminações. Os vídeos foram gravados de forma com que diversas rotações e posições de um mesmo selo fossem obtidos. Também, foi indispensável a ajuda de outras pessoas na gravação desses vídeos, gerando diversidade suficiente para um modelo proveitoso. É fundamental destacar que houve revisão de cada uma das imagens, visando excluir, principalmente, a presença de registros borrados.


Para baixar o dataset gerado pela equipe:

In [ ]:
TEAM_DATASET_URL = "https://drive.google.com/uc?id=1Fm-yUt27Lclx0wl3ngouJZbw34wtWT-F"
TEAM_OUTPUT_DATASET = "dataset-team.zip"

In [ ]:
!gdown "{TEAM_DATASET_URL}" -O "{TEAM_OUTPUT_DATASET}"

Para extrair e fundir o dataset da equipe com o original:

In [ ]:
shutil.unpack_archive(TEAM_OUTPUT_DATASET)
os.remove(TEAM_OUTPUT_DATASET)
delete_useless()

#### 4.2 Remoção das imagens duplicadas ou extremamente semelhantes

A partir dessa etapa de pré-processamento, criaremos uma pasta para ficar o dataset pré-processado.

In [ ]:
FINAL_DATASET_FOLDER = "final_dataset"
final_dataset_path = os.path.join(FINAL_DATASET_FOLDER)

os.makedirs(final_dataset_path, exist_ok=True)

Como vimos anteriormente, a quantidade de imagens duplicadas ou semelhantes no dataset original é muito grande. Para resolver esse problema, iremos remover essas imagens na célula de código abaixo. É importante ressaltar que as imagens do dataset da equipe também serão removidas se forem duplicatas ou semelhantes.

In [ ]:
total_count = 0
removed_images_count = 0

for signal in os.listdir(DATASET_PATH):
    if signal not in SIGNALS:
        continue

    signal_path = os.path.join(DATASET_PATH, signal)
    final_signal_path = os.path.join(final_dataset_path, signal)

    os.makedirs(final_signal_path, exist_ok=True)

    seen_hashes = set()

    for img_filename in os.listdir(signal_path):
        img_lower_filename = img_filename.lower()

        if not (img_lower_filename.endswith(".jpg") or img_lower_filename.endswith(".png") or img_lower_filename.endswith(".jpeg")):
            continue
        
        total_count += 1
        img_path = os.path.join(signal_path, img_filename)
        
        try:
            with Image.open(img_path) as img:
                img_hash = imagehash.phash(img)

                is_simmilar_or_duplicated = False

                for seen_hash in seen_hashes:
                    if seen_hash - img_hash <= IMG_DIFFERENCE_LIMIT:
                        is_simmilar_or_duplicated = True
                        break

                if is_simmilar_or_duplicated:
                    removed_images_count += 1
                    continue

                final_dataset_img_path = os.path.join(final_signal_path, img_filename)

                shutil.copy(img_path, final_dataset_img_path)
                seen_hashes.add(img_hash)
        except (FileNotFoundError, OSError):
            pass

print(f"Total de imagens: {total_count}\nImagens removidas: {removed_images_count}\nImagens restantes: {total_count - removed_images_count}")

O conjunto de imagens que compõe o dataset final está em `final_dataset`.

#### 4.3 Redimensionamento das imagens

Antes de irem para o modelo, as imagens utilizadas foram recortadas em torno das mãos, utilizando a biblioteca MediaPipe, e redimensionadas para uma matriz fixa de pixels, com o objetivo de uniformizar os dados de entrada. Dessa forma, se o MediaPipe não reconhecer pelo menos uma mão na imagem, a imagem é descartada, eliminando capturas de baixa qualidade. Uma consequência positiva do redimensionamento, é a normalização da intensidade dos pixels como visto anteriormente no histograma de intensidade de pixels.

Primeiro, devemos instalar as bibliotecas utilizadas nessa etapa.

In [ ]:
%pip install opencv-python mediapipe

Com as novas versões do MediaPipe, é necessário instalar o modelo:

In [ ]:
!wget -q https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task

Agora, precisamos importar as bibliotecas necessárias e configurar o MediaPipe para reconhecer até duas mãos por foto.

In [ ]:
# importação
import cv2
import numpy as np

import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

In [ ]:
# configuração do MediaPipe
base_options = python.BaseOptions(model_asset_path='hand_landmarker.task')
options = vision.HandLandmarkerOptions(base_options=base_options,
                                       num_hands=2)
detector = vision.HandLandmarker.create_from_options(options)

Afim de escalabilidade, definimos a função que detecta as mãos, cropa a imagem em torno delas, e retorna a matriz de pixels da nova imagem redimensionada.

In [ ]:
TARGET_SIZE = 128
MIN_CROP_SIZE = 50
CROP_PERCENTAGE_PADDING = 1.0

In [ ]:
def process_two_hands_image(image_path):
    img = cv2.imread(image_path)
    if img is None:
        return None
        
    h, w, _ = img.shape

    img_rgb = mp.Image(image_format=mp.ImageFormat.SRGB, data=cv2.cvtColor(img, cv2.COLOR_BGR2RGB))

    detection_result = detector.detect(img_rgb)

    # se não achar mãos, descarta a imagem
    if not detection_result.hand_landmarks:
        return None 

    x_coords = []
    y_coords = []

    # obtém os limites, em coordenadas, das mãos nas imagens
    for hand_landmarks in detection_result.hand_landmarks:
        for lm in hand_landmarks: 
            x_coords.append(int(lm.x * w))
            y_coords.append(int(lm.y * h))

    # obtém os limites iniciais das coordenadas detectadas
    x_min, x_max = min(x_coords), max(x_coords)
    y_min, y_max = min(y_coords), max(y_coords)

    # encontra o ponto central exato da detecção
    center_x = (x_min + x_max) // 2
    center_y = (y_min + y_max) // 2

    # descobre qual é a maior dimensão (largura ou altura)
    box_w = x_max - x_min
    box_h = y_max - y_min
    max_dim = max(box_w, box_h)

    # aplica a margem de segurança (baseada na maior dimensão)
    side_length = int(max_dim * (1 + CROP_PERCENTAGE_PADDING))
    half_side = side_length // 2

    # recalcula as coordenadas baseadas no centro para formar um quadrado 1:1
    new_x_min = center_x - half_side
    new_y_min = center_y - half_side
    new_x_max = center_x + half_side
    new_y_max = center_y + half_side

    # previne que a caixa imaginária saia dos limites da foto original
    # se a mão estiver muito perto da borda, o crop será cortado
    new_x_min = max(0, new_x_min)
    new_y_min = max(0, new_y_min)
    new_x_max = min(w, new_x_max)
    new_y_max = min(h, new_y_max)

    # faz o crop usando as novas coordenadas centralizadas
    crop = img[new_y_min:new_y_max, new_x_min:new_x_max]
    crop_h, crop_w, _ = crop.shape

    if crop_h < MIN_CROP_SIZE or crop_w < MIN_CROP_SIZE:
        return None

    # calcular quanto de borda preta existirá em cada lado da imagem
    # Usamos side_length que é o tamanho desejado do quadrado
    pad_top = (side_length - crop_h) // 2
    pad_bottom = side_length - crop_h - pad_top
    pad_left = (side_length - crop_w) // 2
    pad_right = side_length - crop_w - pad_left

    # adicionar as bordas pretas
    squared_crop = cv2.copyMakeBorder(
        crop, pad_top, pad_bottom, pad_left, pad_right,
        cv2.BORDER_CONSTANT, value=[0, 0, 0]
    )

    # redimensiona para a matriz final
    # Usamos side_length como a maior dimensão original
    interpolation = cv2.INTER_AREA if side_length > TARGET_SIZE else cv2.INTER_CUBIC
    final_matrix = cv2.resize(squared_crop, (TARGET_SIZE, TARGET_SIZE), interpolation=interpolation)

    return final_matrix

O trecho abaixo realiza a transformação em `final_dataset`.

In [ ]:
removed_images_count = 0

for signal in os.listdir(final_dataset_path):
    if signal not in SIGNALS:
        continue

    signal_path = os.path.join(final_dataset_path, signal)

    for img_filename in os.listdir(signal_path):
        img_lower_filename = img_filename.lower()

        if not (img_lower_filename.endswith(".jpg") or img_lower_filename.endswith(".png") or img_lower_filename.endswith(".jpeg")):
            continue
        
        img_path = os.path.join(signal_path, img_filename)

        processed_img_matrix = process_two_hands_image(img_path)

        if processed_img_matrix is None:
            os.remove(img_path)
            removed_images_count += 1
            continue

        cv2.imwrite(img_path, processed_img_matrix)

print(f"{removed_images_count} imagens foram removidas.")

Aqui, mais 214 imagens foram removidas porque o MediaPipe não conseguiu identificar uma mão na imagem. As imagens geradas possuem uma resolução de 128x128, suficiente para visualização das imagens. O formato do corte é baseado no limite de maior tamanho, que é replicado para o lado menor a partir do centro do eixo. Para garantir que as imagens possuem 128px para cada lado, caso o selo esteja na borda da imagem, uma borda preta é colocada.

[OPCIONAL] Para visualizar a forma atual das imagens, podemos plotar novamente as imagens de cada selo de mão no dataset.

In [ ]:
plot_by_signal_images(final_dataset_path)

#### 4.4 Conversão para escala de cinza

Para reduzir a quantidade de informação das imagens, elas serão convertidas para uma escala de cinza. Além disso, essa conversão é fundamental para garantir que o modelo esteja livre de viés em relação à cor de pele de quem está na imagem.

In [ ]:
before_bytes = 0
after_bytes = 0

for signal in os.listdir(final_dataset_path):
    if signal not in SIGNALS:
        continue

    signal_path = os.path.join(final_dataset_path, signal)

    for img_filename in os.listdir(signal_path):
        img_lower_filename = img_filename.lower()

        if not (img_lower_filename.endswith(".jpg") or img_lower_filename.endswith(".png") or img_lower_filename.endswith(".jpeg")):
            continue
        
        img_path = os.path.join(signal_path, img_filename)

        before_bytes += os.path.getsize(img_path)

        img = cv2.imread(img_path)
        grey_img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

        cv2.imwrite(img_path, grey_img)

        after_bytes += os.path.getsize(img_path)

reduce_percent = ((before_bytes - after_bytes) / before_bytes) * 100
print(f"Bytes antes: {before_bytes}\nBytes depois: {after_bytes}\nRedução do tamanho do dataset: {reduce_percent:.2f}")

Como resultado, além da evidente queda na complexidade da matriz de pixels em memória (66,66% já que, de 3 canais de cor, fomos para 1), o tamanho do dataset em escala de cinza é 27,59% menor do que o original. A redução não é de 66,66% nesse caso porque os algoritmos de compressão das imagens já reduzem-as ao armazenar.

[OPCIONAL] Para observar os selos em escala de cinza, rode a célula abaixo.

In [ ]:
plot_by_signal_images(final_dataset_path, "gray")

#### 4.5 Planificação das matrizes

Nessa última etapa do pré-processamento, as matrizes de pixels foram reorganizadas de modo que se tornassem matrizes unidimensionais, facilitando a utilização delas nos algoritmos requisitados.

In [ ]:
import os
import cv2
import numpy as np

In [ ]:
X_list = [] # features
y_list = [] # labels

for signal in os.listdir(final_dataset_path):
    if signal not in SIGNALS:
        continue
        
    signal_path = os.path.join(final_dataset_path, signal)
    
    for img_filename in os.listdir(signal_path):
        img_lower = img_filename.lower()
        if not (img_lower.endswith(".jpg") or img_lower.endswith(".png") or img_lower.endswith(".jpeg")):
            continue
            
        img_path = os.path.join(signal_path, img_filename)
        
        img_gray = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        
        if img_gray is not None:
            img_flattened = img_gray.flatten()
            
            X_list.append(img_flattened)
            y_list.append(signal)

# converte para matrizes numpy
X = np.array(X_list)
y = np.array(y_list)

print("--- DIMENSÕES DOS DADOS PLANIFICADOS ---")
print(f"Formato da Matriz de Features (X): {X.shape}")
print(f"Formato do Vetor de Alvos (y): {y.shape}")

# 5. Separação dos Dados

Os dados foram separados com a seguinte proporção: 80% para os dados de treinamento e 20% para os dados de teste. Essa proporção foi escolhida por ser a padrão para treinamento de classificação de imagens para a quantidade de dados que possuímos.

Nenhuma técnica específica para estratificação dos dados foi utilizada. A única garantia ao usar o `train_test_split` com `stratify` é que as classes possuem a mesma proporção nos dados para treinamento e nos dados de teste.

<!-- Para seguir essa divisão, os dados são divididos em subpastas dentro dos sinais de mão, onde cada subpasta é nomeada como uma identificador único por vídeo de origem. Assim, 80% das subpastas são utilizadas para treinamento e 20% para teste. Isso evita que imagens de um mesmo vídeo sejam utilizadas para teste e treinamento simultânemante, o que poderia gerar uma taxa de acerto alta artificialmente devido à semelhança dos dados.  -->

Primeiro, devemos baixar a biblioteca `scikit-learn`.

In [ ]:
%pip install scikit-learn

Segundo, precisamos importar os módulos utilizados.

In [ ]:
from sklearn.model_selection import train_test_split
import numpy as np

Por fim, aplicamos a estratificação dos dados.

In [ ]:
TEST_PERCENT = 0.20

X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=TEST_PERCENT, 
    random_state=42, 
    stratify=y
)

print("--- DISTRIBUIÇÃO DOS CONJUNTOS ---")
print(f"Treino: {X_train.shape[0]} amostras ({100 - TEST_PERCENT*100:.0f}%)")
print(f"Teste: {X_test.shape[0]} amostras ({TEST_PERCENT*100:.0f}%)")

print("\n--- DISTRIBUIÇÃO DE CLASSES NO TESTE ---")
unique, counts = np.unique(y_test, return_counts=True)
for label, count in zip(unique, counts):
    print(f"{label}: {count} imagens")

Como pode ser visto no trecho acima, 80% das amostas foram redirecionadas para o conjunto de treinamento, já 20% para o conjunto de testes. A distribuição de classes no teste demonstra que a proporção 80/20 foi recolhida com base em cada selo de mão, não de forma cega no mundo de todas as imagens. Aqui, também é possível observar a diferença na quantidade de imagens dos selos.

# 6. Modelagem

Para cada um dos modelos será exibido um relatório sobre seus resultados. As discussões estarão na seção seguinte (7).

#### 6.1 DummyClassifier (Baseline)

O modelo abaixo apenas tenta chutar o selo de mão. Ele será usado para demonstrar como a utilização de um método de classificação inteligente é superior.

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, classification_report

dummy_model = DummyClassifier(strategy="most_frequent")

# treinamento
dummy_model.fit(X_train, y_train)

# testes
y_pred_dummy = dummy_model.predict(X_test)

accuracy_dummy = accuracy_score(y_test, y_pred_dummy)
print(f"Acurácia do Baseline (Chute Cego): {accuracy_dummy * 100:.2f}%")

##### Parâmetros principais

* `strategy='most_frequent'`: Define que o modelo deve chutar o selo de mão com maior frequência até aquele momento sempre.

#### 6.2 SGDClassifier

Já o trecho seguinte treina e testa o modelo SGDClassifier com função de custo hinge:

In [ ]:
from sklearn.linear_model import SGDClassifier

sgd_model = SGDClassifier(random_state=42, max_iter=1000, tol=1e-3, n_jobs=-1)

print("Iniciando o treinamento do SGD Classifier...")

# treinamento
sgd_model.fit(X_train, y_train)

print("Treinamento concluído!\n")

# testes
y_pred_sgd = sgd_model.predict(X_test)

accuracy_sgd = accuracy_score(y_test, y_pred_sgd)

print("--- RESULTADOS: SGD CLASSIFIER ---")
print(f"Acurácia Geral: {accuracy_sgd * 100:.2f}%\n")
print("Relatório de Classificação Detalhado:")
print(classification_report(y_test, y_pred_sgd))

##### Parâmetros principais

* `loss='hinge'` (valor padrão): Define a função de custo (perda) que o otimizador tentará minimizar. Utilizar hinge faz com que o algoritmo opere como uma Máquina de Vetores de Suporte Linear (Linear SVM), buscando encontrar um hiperplano ótimo que maximize a margem de separação entre as classes de selos.

* `max_iter=1000`: Define o limite máximo de iterações. Atua como uma trava de segurança para garantir que o treinamento seja interrompido, evitando ciclos infinitos caso o gradiente não consiga encontrar um ponto de convergência absoluto no espaço de 16.384 dimensões.

* `tol=1e-3` (0.001): Finaliza o algoritmo antes da quantidade máxima de iterações caso a redução de erro seja inferior a 0,001% entre iterações.

* `n_jobs=-1`: Garante que o modelo utilize a maior quantidade de núcleos de processamento possível.

#### 6.3 RandomForestClassifier

O trecho a seguinte treina e testa um modelo de `RandomForestClassifier` do `scikit-learn` com 500 árvores de decisão:

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(n_estimators=500, random_state=42, n_jobs=-1)

# treinamento
print("Iniciando o treinamento do Random Forest...")

rf_model.fit(X_train, y_train)

print("Treinamento concluído!\n")

# testes
y_pred_rf = rf_model.predict(X_test)

accuracy_rf = accuracy_score(y_test, y_pred_rf)

print("--- RESULTADOS: RANDOM FOREST ---")
print(f"Acurácia Geral: {accuracy_rf * 100:.2f}%\n")
print("Relatório de Classificação Detalhado:")
print(classification_report(y_test, y_pred_rf))

##### Parâmetros utilizados

* `n_estimators=500`: Define o tamanho da floresta, ou seja, o algoritmo criará 500 árvores de decisão distintas. Nós testamos utilizar 100 árvores de decisão, o que causou uma precisão 2% menor (quase irrelevante). Porém, como aumentar a quantidade de árvores não acresceu tanto o tempo de execução, optamos por utilizar um maior número de árvores de decisão afim de manter a estabilidade dos resultados.

* `random_state=42`: Fixar esse valor permite que os mesmos resultados possam ser reproduzíveis com a mesma seed.

* `n_jobs=-1`: Garante que o modelo utilize a maior quantidade de núcleos de processamento possível.

#### 6.4 Comparação entre resultados

O modelo Dummy Classifier (Baseline) estabeleceu o limite inferior de acerto em 11.07%. O SGDClassifier atingiu uma acurácia geral de 32.06%, enquanto o RandomForestClassifier demonstrou superioridade clara, alcançando 51.53% de acurácia global.

#### 6.5 Modelo final

A escolha do RandomForestClassifier como modelo final definitivo para este projeto justifica-se pelo seu desempenho superior. A arquitetura de múltiplos estimadores não-lineares provou-se mais adequada para mapear a alta dimensionalidade dos dados (16.384 atributos) do que a abordagem de separação linear estrita adotada pelo SGD.

# 7. Avaliação e Discussão

#### 7.1 Matrizes de confusão

Só serão geradas as matrizes de confusão dos modelos inteligentes, já que analisar a do Dummy Classifier não geraria nenhuma conclusão relevante.

A célula seguinte plota a matriz de confusão do SGD Classifier.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

fig, ax = plt.subplots(figsize=(10, 8))
ConfusionMatrixDisplay.from_predictions(
    y_test, 
    y_pred_sgd, 
    cmap='Blues', 
    ax=ax, 
    xticks_rotation=45
)

plt.title("Matriz de Confusão - SGD Classifier")
plt.tight_layout()
plt.show()

Já a próxima célula, gera a matriz do Random Forest Classifier.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

fig, ax = plt.subplots(figsize=(10, 8))
ConfusionMatrixDisplay.from_predictions(
    y_test, 
    y_pred_rf, 
    cmap='Blues', 
    ax=ax, 
    xticks_rotation=45
)

plt.title("Matriz de Confusão - Random Forest Classifier")
plt.tight_layout()
plt.show()

#### 7.2 Avaliação e Discussão

##### Avaliação de Desempenho Global

A avaliação do modelo vencedor, o RandomForestClassifier (51.53% de acurácia), foi pautada na análise conjunta de sua matriz de confusão e das métricas de precisão, revocação e F1-score, comparando-as com o desempenho do SGDClassifier (32.06%). O Random Forest apresentou um comportamento preditivo muito mais robusto e consistente (F1-score Macro de 0.50), superando tranquilamente o limite inferior estabelecido pela linha de base do Dummy Classifier (11.07%).

##### Análise das Matrizes de Confusão

As matrizes de confusão revelam visualmente a disparidade de aprendizado entre os dois algoritmos e evidenciam problemas no dataset.

No SGD Classifier, nota-se uma matriz altamente dispersa, sem uma diagonal principal forte. O modelo falhou criticamente na generalização, o que é evidenciado pela incapacidade total de classificar o selo "rat" (0 acertos) e pelo acerto de apenas 1 amostra para o selo "ox".

No Random Forest Classifier, a diagonal principal é expressivamente mais densa, destacando a alta capacidade do modelo em identificar classes como "horse", com 16 acertos, "dragon", com também 16 acertos, e "monkey", com 15. Contudo, a matriz também expõe os falsos positivos e negativos que comprometeram a acurácia global: a classe "snake" foi erroneamente classificada como "tiger" em 7 ocasiões, e as classes "rat" e "ox" permaneceram sendo as classes com as maiores taxas de erro, com apenas 3 e 2 acertos, respectivamente.

##### Limitações Estruturais

O fato de ambos os classificadores (um linear e um não-linear) errarem majoritariamente nas exatas mesmas classes ("ox" e "rat") sugere fortemente que a limitação não reside na capacidade matemática dos algoritmos, mas na natureza dos dados extraídos, os quais, como já apontados no início do notebook, possuem disparidades gritantes na quantidade de dados para cada selo de mão. Além disso, a transformação das imagens para a escala de cinza, seguida de sua planificação em vetores unidimensionais de 16 mil colunas, reduz a percepção espacial e de profundidade tridimensional das mãos, o que pode explicar parte dos erros.